In [ ]:
import pandas as pd

# Load the dataset
datadf = pd.read_csv(
    '/content/u.data',
    sep='\t',
    names=['user_id', 'item_id', 'rating', 'timestamp']
)
datadf = datadf [['user_id', 'item_id', 'rating']]
print(datadf.head())

   user_id  item_id  rating
0      196      242       3
1      186      302       3
2       22      377       1
3      244       51       2
4      166      346       1


In [ ]:
def laplace_mechanism(matrix, epsilon, mean_rating):
    sensitivity = 4.0
    noise = np.random.laplace(0, sensitivity/epsilon, matrix.shape)
    noisy_matrix = matrix + noise
    noisy_matrix = np.clip(noisy_matrix, 1, 5)
    return noisy_matrix

In [ ]:
import numpy as np

def piecewise_mechanism(x, epsilon):
    # Step 1: normalise x from [1,5] to [-1,1]
    x_norm = 2 * (x - 1) / (5 - 1) - 1
    # C = (e^(ε/2) + 1) / (e^(ε/2) - 1)
    c= (np.exp(epsilon/2)+1)/(np.exp(epsilon/2)-1)
    # p = (e^ε - e^(ε/2)) / (2 * e^(ε/2) + 2)
    p= (np.exp(epsilon)-np.exp(epsilon/2))/(2*np.exp(epsilon/2)+2)
    # l(x) = (C+1)/2 * x_norm - (C-1)/2
    # r(x) = l(x) + C - 1
    l= (c+1)/2*x_norm-(c-1)/2
    r=l+c-1
    # threshold = e^(ε/2) / (e^(ε/2) + 1)
    threshold = np.exp(epsilon/2)/(np.exp(epsilon/2)+1)
    if np.random.uniform(0, 1) < threshold:
    # heads — sample from centre zone
      output = np.random.uniform(l, r)
    else:
    # tails — sample from tails
    # left tail runs from -C to l
    # right tail runs from r to +C
    # pick one randomly
      output = np.random.uniform(-c, l) if np.random.uniform(0,1) < 0.5 else np.random.uniform(r, c)
    x_out = (output + 1) * 2 + 1
    x_out = np.clip(x_out, 1, 5)
    return x_out

In [ ]:
def apply_piecewise(matrix, epsilon, mean_rating):
    noisy_matrix = matrix.copy()
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] != mean_rating:
                noisy_matrix[i, j] = piecewise_mechanism(matrix[i, j], epsilon)
    return noisy_matrix

In [ ]:
def gaussian_mechanism(matrix, epsilon, delta, mean_rating):
    sensitivity = 4.0
    sigma = sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    noise = np.random.normal(0, sigma, matrix.shape)
    noisy_matrix = matrix + noise
    noisy_matrix = np.clip(noisy_matrix, 1, 5)
    return noisy_matrix

In [ ]:
def bounded_laplace_mechanism(matrix, epsilon, mean_rating):
    sensitivity = 1.0  # normalised scale [0,1]
    b = sensitivity / epsilon
    noisy_matrix = matrix.copy()
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] != mean_rating:
                r = (matrix[i, j] - 1) / 4  # normalise to [0,1]
                while True:
                    noise = np.random.laplace(0, b)
                    r_noisy = r + noise
                    if 0 <= r_noisy <= 1:
                        break
                noisy_matrix[i, j] = r_noisy * 4 + 1  # denormalise
    return noisy_matrix

In [ ]:
def subsample_ratings(df, keep_fraction, random_state=42):
    return df.sample(frac=keep_fraction, random_state=random_state)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error


sparsity_levels = {'natural': 1.0, '50%': 0.5, '20%': 0.2}
epsilons = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sparsity_results = []

for sparsity_name, fraction in sparsity_levels.items():
    # subsample
    df_sparse = subsample_ratings(datadf, fraction)

    # split and build matrix
    train_s, test_s = train_test_split(df_sparse,
                                       test_size=0.2,
                                       random_state=42)
    train_matrix_s = train_s.pivot(index='user_id',
                                   columns='item_id',
                                   values='rating')
    train_array_s = train_matrix_s.values.astype(float)
    mean_rating_s = np.nanmean(train_array_s)
    train_array_s[np.isnan(train_array_s)] = mean_rating_s

    test_users_s = test_s['user_id'].values
    test_items_s = test_s['item_id'].values
    test_ratings_s = test_s['rating'].values
    gaussian_with_delta = lambda matrix, eps, mean_rating: gaussian_mechanism(matrix, eps, 1e-5, mean_rating)
    for eps in epsilons:                                        # ← epsilon loop
        for mechanism_name, mechanism_fn in [
                      ('clipped_laplace', laplace_mechanism),
                      ('bounded_laplace', bounded_laplace_mechanism),
                      ('gaussian', gaussian_with_delta),
                      ('piecewise', apply_piecewise)]:
            noisy = mechanism_fn(train_array_s, eps, mean_rating_s)  # ← pass mean_rating_s

            svd = TruncatedSVD(n_components=20, random_state=42)
            reduced = svd.fit_transform(noisy)
            predicted_matrix = np.dot(reduced, svd.components_)

            predicted_ratings = []
            for user, item in zip(test_users_s, test_items_s):
                if user in train_matrix_s.index and item in train_matrix_s.columns:
                    u_idx = train_matrix_s.index.get_loc(user)
                    i_idx = train_matrix_s.columns.get_loc(item)
                    predicted_ratings.append(predicted_matrix[u_idx, i_idx])
                else:
                    predicted_ratings.append(mean_rating_s)

            rmse = np.sqrt(mean_squared_error(test_ratings_s, predicted_ratings))
            sparsity_results.append({
                'sparsity': sparsity_name,
                'epsilon': eps,
                'mechanism': mechanism_name,
                'rmse': rmse
            })
            print(f"Sparsity: {sparsity_name}, ε: {eps}, {mechanism_name}: {rmse:.4f}")

Sparsity: natural, ε: 0.1, clipped_laplace: 1.3110
Sparsity: natural, ε: 0.1, bounded_laplace: 1.1653
Sparsity: natural, ε: 0.1, gaussian: 1.3248
Sparsity: natural, ε: 0.1, piecewise: 1.1565
Sparsity: natural, ε: 0.5, clipped_laplace: 1.2630
Sparsity: natural, ε: 0.5, bounded_laplace: 1.1527
Sparsity: natural, ε: 0.5, gaussian: 1.3155
Sparsity: natural, ε: 0.5, piecewise: 1.1375
Sparsity: natural, ε: 1.0, clipped_laplace: 1.2162
Sparsity: natural, ε: 1.0, bounded_laplace: 1.1407
Sparsity: natural, ε: 1.0, gaussian: 1.2987
Sparsity: natural, ε: 1.0, piecewise: 1.1128
Sparsity: natural, ε: 2.0, clipped_laplace: 1.1660
Sparsity: natural, ε: 2.0, bounded_laplace: 1.1248
Sparsity: natural, ε: 2.0, gaussian: 1.2766
Sparsity: natural, ε: 2.0, piecewise: 1.0746
Sparsity: natural, ε: 5.0, clipped_laplace: 1.1062
Sparsity: natural, ε: 5.0, bounded_laplace: 1.0860
Sparsity: natural, ε: 5.0, gaussian: 1.2181
Sparsity: natural, ε: 5.0, piecewise: 1.0454
Sparsity: natural, ε: 10.0, clipped_laplace: 